<a href="https://colab.research.google.com/github/clark-julia/econ5200-PS1/blob/main/01_positioning_and_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!git clone https://github.com/clark-julia/econ5200-midterm-replication.git
%cd /content/econ5200-midterm-replication
!python download_data.py

Cloning into 'econ5200-midterm-replication'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 82 (delta 37), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 3.37 MiB | 3.71 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/econ5200-midterm-replication
Hansen (2015), punishment and deterrence in drunk driving
  data/hansen_dwi.dta  9.5 MB


# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

## Data

In [ ]:
# Import fetch and PAPERS from download_data.py so they are defined in this scope
from download_data import fetch, PAPERS
MY_PAPER = "hansen"

# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    # TODO: read the file(s), e.g. df = pd.read_stata(paths[0])
    dwi = pd.read_stata(paths[0])
    print(dwi.shape)
    print(dwi["year"].min(), dwi["year"].max())
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving
(214558, 12)
1999.0 2007.0


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. de Figueiredo, M. F. P. (2015). Throw away the jail or throw away the key? The effect of punishment on recidivism and social cost. Arizona State Law Journal.

2. Sloan, F., Eldred, L., McCutchan, S., & Platt, A. (2016). Deterring rearrests for drinking and driving. Southern Economic Journal, 83(2), 416–436.

3. Kolesár, M., & Rothe, C. (2018). Inference in regression discontinuity designs with a discrete running variable. American Economic Review, 108(8), 2277–2304.

**The identification strategy and its assumption, in my own words:**

Hansen wants to know whether harsher punishment deters drunk driving, but you can't just compare harshly and lightly punished drivers, because the ones with higher BACs are different to begin with. So he uses the legal cutoffs instead. In Washington, a BAC of 0.08 triggers a DUI and 0.15 triggers an aggravated DUI with tougher penalties, so punishment jumps at each line while the drivers on either side look basically the same. He compares recidivism just below and just above each cutoff using local linear regressions. The key assumption is that nothing else changes abruptly at the cutoff, so it's essentially luck which side of the line a driver lands on. He supports this with smooth density tests and balanced driver characteristics at both thresholds.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [ ]:
# YOUR TURN — Section 2, no AI: set RUNNING, H, KERNEL, CONTROLS, fit rd_fit, and set JUMP
CUTOFF = 0.08
dwi["bac_min"] = dwi["low_score"].astype(float) / 1000

RUNNING = "bac_min"
H = 0.05
KERNEL = "rectangular"
CONTROLS = ["male", "white", "aged", "acc"]

s = dwi[(dwi[RUNNING] >= CUTOFF - H) & (dwi[RUNNING] <= CUTOFF + H)].copy()
s["x"] = s[RUNNING] - CUTOFF
s["dui"] = (s[RUNNING] >= CUTOFF).astype(int)

#controls
rd_fit = smf.ols("recidivism ~ dui + x + dui:x + male + white + aged + acc + C(year)", data=s).fit(cov_type="cluster", cov_kwds={"groups": s["low_score"]})
JUMP = float(rd_fit.params["dui"])
print("Controls:", JUMP, rd_fit.bse["dui"], len(s))

#No controls
rd_fit_no_controls = smf.ols("recidivism ~ dui + x + dui:x", data=s).fit(cov_type="cluster", cov_kwds={"groups": s["low_score"]})
JUMP_NO_CONTROLS = float(rd_fit_no_controls.params["dui"])
print("No controls:", JUMP_NO_CONTROLS, rd_fit_no_controls.bse["dui"], len(s))

results_table = pd.DataFrame({
    "testn": ["With Controls", "Without Controls"],
    "Jump": [rd_fit.params["dui"], rd_fit_no_controls.params["dui"]],
    "SE": [rd_fit.bse["dui"], rd_fit_no_controls.bse["dui"]],
    "P-value": [rd_fit.pvalues["dui"], rd_fit_no_controls.pvalues["dui"]],
    "N": [len(s), len(s)]
})

# Format table
results_table["Jump"] = results_table["Jump"].map("{:.6f}".format)
results_table["SE"] = results_table["SE"].map("{:.6f}".format)
results_table["P-value"] = results_table["P-value"].map("{:.6f}".format)
results_table["N"] = results_table["N"].map("{:,}".format)

display(results_table)

Controls: -0.022163327949850046 0.004141808892258096 93899
No controls: -0.021845285314709577 0.004174708738842859 93899


,testn,Jump,SE,P-value,N
0,With Controls,-0.022163,0.004142,0.000000,"93,899"
1,Without Controls,-0.021845,0.004175,0.000000,"93,899"


### Your number, with its unit

`recidivism` is 0 or 1, so its mean is a proportion and your jump is a change in a proportion. Read it in
**percentage points**: −0.02 is minus 2 percentage points, not "minus 2 percent". If you want a relative change,
name its base: the mean recidivism just below the cutoff, within your bandwidth. Fill all five names:

- `PRINTED`: the jump exactly as your code prints it
- `IN_UNITS`: the same jump in the unit you name in `UNIT`
- `UNIT`: one of `percentage points`, `percent relative to the mean below the cutoff`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = JUMP
IN_UNITS = 100* JUMP
UNIT = "percentage points"
PAPER_NUMBER = -0.021
PAPER_WHERE = "Table 3, panel A (bandwidth 0.05), all tested driver, with controls"

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

if RUNNING is None or H is None:
    BASE = None
else:
    below = dwi[(dwi[RUNNING] < CUTOFF) & (dwi[RUNNING] >= CUTOFF - H)]
    BASE = below["recidivism"].mean()                  # the mean just below the cutoff, within your bandwidth
HANSEN_UNITS = {
    "percentage points": [lambda b: 100 * b],
    "percent relative to the mean below the cutoff": [lambda b: 100 * b / BASE if BASE else float("nan")],
}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, HANSEN_UNITS)

Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.


### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

# One line per row; take my_estimate from the cell that computed it, never type it in.
headline.loc["Table 3, Panel A (bandwidth 0.05): Impact of BAC over 0.08 on recidivism"] = [
    -0.021,
    JUMP,
    np.nan,
    "Limited sample with no county or prior-offense controls"
]
headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
"Table 3, Panel A (bandwidth 0.05): Impact of BAC over 0.08 on recidivism",-0.021,-0.022163,-5.539657,Limited sample with no county or prior-offense...


### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | Table 3 Panel A | My value was -0.0222 and the paper was -0.021| 0.0012 | the specific sample and control variables | The window, cutoff, kernel, and clustering were identical to the paper | no |
| 2 | Table 3 Panel A | The paper controls for county FE and other previus offenses | The value was -0.0218 with no controls vs the -0.0222 | the variable was not thoroughly defined in the public data | Ran without controls and observed almost no change | No |
| 3 | Table 3 Panel A | My sample size was 92899 and the paper was 95111 | 1212 less stops | the data was probably cleaned differently | The BAC and years were the same but N differed | Yes, the methods for cleaning the data is missing |
| 4 | Table 3 Panel A | BAC reading, bandwidth kernal and controls | No gap  | The conclusion from the paper | The paper used the lower reading so I used the low_score bandwidth, rectangular kernal like in table 3 | No |


**Ambiguities in the paper, to take into Section 3:**

- Data cleaning methods
- Clusters for the standard errors by BAC
- How the bandwidth was chosen